# OSNet: OOF-признаки → компактный реранкер

Run All в существующем `.venv` запускает весь эксперимент. MVP и старые результаты не меняются.
Три OSNet: по 5 эпох, исходный LR-горизонт 30; затем две маленькие головы по 7 эпох.
GPU выбирается автоматически (CUDA → MPS → CPU). Новых загрузок/установок нет.
Подробный протокол, ограничения и resume: [README.md](README.md).

In [1]:
from pathlib import Path
import sys
import json

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'training/oof_pair_reranker.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Откройте ноутбук внутри Car-classification-MSK')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from training.oof_pair_reranker import EXPERIMENT, prepare, run
OUTPUT = EXPERIMENT / 'results/run_01'
DEVICE = None  # автоматически; можно явно 'mps' / 'cuda' / 'cpu'
print('Python:', sys.executable)
print('Результаты:', OUTPUT)

Python: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python
Результаты: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_13_oof_pair_reranker/results/run_01


## 1. Проверки до обучения

Checksum исходных кадров, identity/frame-disjoint фолды, рецепт MVP и предыдущая голова.
Эта ячейка ничего не обучает и не записывает результаты. На Mac ожидаем `mps`; если выбран `cpu`, проверьте ядро перед следующей ячейкой.

In [2]:
plan = prepare(DEVICE)['plan']
DEVICE = plan['device']
print(json.dumps(plan, ensure_ascii=False, indent=2))

Verify frozen frame hashes and identity/frame separation
OSNet: 16/16
{
  "device": "mps",
  "outer_identities": {
    "train": 925,
    "calibration": 307,
    "validation": 309
  },
  "folds": {
    "fold_01": {
      "train_ids": 616,
      "held_out_ids": 309,
      "queries": 309,
      "gallery": 892,
      "batches_per_epoch": 38
    },
    "fold_02": {
      "train_ids": 617,
      "held_out_ids": 308,
      "queries": 308,
      "gallery": 921,
      "batches_per_epoch": 38
    },
    "fold_03": {
      "train_ids": 617,
      "held_out_ids": 308,
      "queries": 308,
      "gallery": 883,
      "batches_per_epoch": 38
    }
  },
  "encoder_epochs_per_fold": 5,
  "lr_horizon_epochs": 30,
  "head_epochs": 7,
  "free_disk_gib": 19.98,
  "backbone_initializer": "/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/models/osnet_ain_x1_0_vehicle_reid.onnx",
  "mvp_used_as_initializer": false,
  "training_started": false,
  "mvp_spot_check_max_a

## 2. Автоматическое обучение и сравнение

Фолд 1 → фолд 2 → фолд 3 → OOF и matched-контроль → beta на calibration → фиксация → validation → маски/скорость/отчёт.
Ни один отложенный фолд не выбирает эпохи. Порог отказа остаётся прежним.

В выводе: fold, эпоха, loss, время эпохи, оставшиеся эпохи, ETA. После прерывания снова Run All; не удаляйте JSON. Один экземпляр одновременно.
Компьютер должен оставаться включённым и без сна; управление питанием здесь не меняется.

In [3]:
report = run(OUTPUT, device=DEVICE)

Verify frozen frame hashes and identity/frame separation
OSNet: 16/16
{
  "device": "mps",
  "outer_identities": {
    "train": 925,
    "calibration": 307,
    "validation": 309
  },
  "folds": {
    "fold_01": {
      "train_ids": 616,
      "held_out_ids": 309,
      "queries": 309,
      "gallery": 892,
      "batches_per_epoch": 38
    },
    "fold_02": {
      "train_ids": 617,
      "held_out_ids": 308,
      "queries": 308,
      "gallery": 921,
      "batches_per_epoch": 38
    },
    "fold_03": {
      "train_ids": 617,
      "held_out_ids": 308,
      "queries": 308,
      "gallery": 883,
      "batches_per_epoch": 38
    }
  },
  "encoder_epochs_per_fold": 5,
  "lr_horizon_epochs": 30,
  "head_epochs": 7,
  "free_disk_gib": 19.98,
  "backbone_initializer": "/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/models/osnet_ain_x1_0_vehicle_reid.onnx",
  "mvp_used_as_initializer": false,
  "training_started": false,
  "mvp_spot_check_max_a

epoch 1:   0%|          | 0/38 [00:00<?, ?it/s]

fold_01 | epoch 1/5, remaining 4 | epoch 00:00:35 | fold ETA 00:02:18 | all encoders ETA ~00:08:05 | loss 7.91427 | no held-out evaluation


epoch 2:   0%|          | 0/38 [00:00<?, ?it/s]

fold_01 | epoch 2/5, remaining 3 | epoch 00:00:31 | fold ETA 00:01:38 | all encoders ETA ~00:07:06 | loss 6.59254 | no held-out evaluation


...

fold_03 | epoch 3/5, remaining 2 | epoch 00:00:31 | fold ETA 00:01:02 | all encoders ETA ~00:01:02 | loss 4.70299 | no held-out evaluation


epoch 4:   0%|          | 0/38 [00:00<?, ?it/s]

fold_03 | epoch 4/5, remaining 1 | epoch 00:00:31 | fold ETA 00:00:31 | all encoders ETA ~00:00:31 | loss 3.21605 | no held-out evaluation


epoch 5:   0%|          | 0/38 [00:00<?, ?it/s]

fold_03 | epoch 5/5, remaining 0 | epoch 00:00:31 | fold ETA 00:00:00 | all encoders ETA ~00:00:00 | loss 2.23043 | no held-out evaluation
fold_03_oof.npz: 512/1191 | ETA 17.7s
fold_03_oof.npz: 1024/1191 | ETA 4.4s
fold_03_oof.npz: 1191/1191 | ETA 0.0s
fold_03_matched.npz: 512/1191 | ETA 17.0s
fold_03_matched.npz: 1024/1191 | ETA 4.2s
fold_03_matched.npz: 1191/1191 | ETA 0.0s
scalar_linear | epoch 1/7, remaining 6 | epoch 0.0s | ETA <= 0.1s | fixed-budget refit
scalar_linear | epoch 2/7, remaining 5 | epoch 0.0s | ETA <= 0.1s | fixed-budget refit
scalar_linear | epoch 3/7, remaining 4 | epoch 0.0s | ETA <= 0.0s | fixed-budget refit
scalar_linear | epoch 4/7, remaining 3 | epoch 0.0s | ETA <= 0.0s | fixed-budget refit
scalar_linear | epoch 5/7, remaining 2 | epoch 0.0s | ETA <= 0.0s | fixed-budget refit
scalar_linear | epoch 6/7, remaining 1 | epoch 0.0s | ETA <= 0.0s | fixed-budget refit
scalar_linear | epoch 7/7, remaining 0 | epoch 0.0s | ETA <= 0.0s | fixed-budget refit
scalar_linea

## 3. Результаты

Сравнение с простым реранкингом, предыдущей головой и matched-контролем.
Просмотр validation не запускает новый поиск. Продвижение в MVP — отдельное решение.

In [4]:
from IPython.display import Markdown, display
display(Markdown((OUTPUT / 'RESULTS.md').read_text(encoding='utf-8')))

# OOF-реранкер: результаты

MVP и механизм отказа не изменены. Продвижения весов нет.

## Validation

| Система | mAP@10, % | Rank-1, % | F1, % | TNR, % |
|---|---:|---:|---:|---:|
| mvp | 81.4689 | 80.1619 | 72.8606 | 79.0323 |
| osnet_neighbors | 81.7191 | 80.5668 | 72.8606 | 79.0323 |
| oof | 81.8113 | 80.5668 | 72.8606 | 79.0323 |
| matched_in_sample | 81.9056 | 80.9717 | 72.8606 | 79.0323 |
| previous_head | 81.7764 | 80.5668 | 72.8606 | 79.0323 |

## Вклад OOF

- К osnet_neighbors: ΔmAP +0.0922 п.п.; 95% парный интервал [-0.0174; +0.2305] п.п.
- К previous_head: ΔmAP +0.0349 п.п.; 95% парный интервал [-0.0596; +0.1350] п.п.
- К matched_in_sample: ΔmAP -0.0943 п.п.; 95% парный интервал [-0.2231; +0.0227] п.п.

Зафиксированные на calibration веса: `{'oof': 0.1, 'matched_in_sample': 0.1, 'previous_head': 0.1}`.

## Трудность обучающих пар

| Fold | OOF hit@50 | MVP hit@50 (те же q/g) | OOF raw mAP@10 |
|---|---:|---:|---:|
| fold_01 | 0.9919 | 1.0000 | 0.7655 |
| fold_02 | 0.9715 | 0.9959 | 0.7388 |
| fold_03 | 0.9634 | 1.0000 | 0.7020 |

## Ограничения и воспроизводимость

- Три encoder обучены с публичной stock-инициализации на двух фолдах, ровно 5 эпох с LR-горизонтом 30. Отложенный фолд не выбирает эпоху и не обновляет BatchNorm.
- OOF-голова: 8 скалярных признаков, 7 фиксированных эпох. Она НЕ переобучается затем на seen-признаках MVP.
- matched_in_sample: та же архитектура, seed, число эпох и q/g, но признаки MVP. Отличаются также сами encoder и число обучающих identity; это не идеально изолированный причинный эксперимент.
- Векторы разных encoder не объединяются; объединяются только скалярные признаки независимо построенных top50.
- Порог и confidence прежние. TNR/решения принять-отказать должны совпадать; F1 может меняться из-за нового top1.
- Outer validation уже использовалась в истории проекта. Bootstrap по query с фиксированной gallery не учитывает историю подбора.
- Ни camera/identity, ни другие query не являются входом реранкера. Позитивы вне raw top50 не восстанавливаются.
- Основной эксперимент без масок; исправленные ручные маски используются только для диагностики с прежним порогом.
- Веса deployment (MVP + одна OOF-голова): 8.35 MiB. Три fold-encoder в deployment не нужны.
- CPU median/p95 после эмбеддинга: 0.317/0.633 ms. Это не официальный RTX A5000 benchmark; extract не меняется.
- Полный отчёт, mask audit, timing и SHA: report.json. Проверенные организаторским evaluator CSV: validation/.
- Сначала оцениваем прирост порядка +1 п.п. без ухудшения отказа; автоматически никто в MVP не внедряется.
